# ISEL - Processamento de Imagem e Visão(PIV)
## Semestre de Inverno 2025/26
# <br>
###  <center> Trabalho Prático 1 </center>
###  <center> Trabalho nº 2.A – Contagem de Veículos em Imagens de Vídeo </center>
# </br>



Trabalho realizado por:


| Aluno | Número |
|-------|--------|
| **Ruben Zhang** | A51388 |
| **Rodrigo Neves** | A51452 |


<b> Turma 51D - Docente: Pedro Jorge </b>

<b>Objetivos:</b>


<b>Descrição:</b>


In [ ]:
#Imports necessários ao longo do trabalho
import numpy as np
import matplotlib.pyplot as plt
import cv2


In [ ]:

def AbrirVideo(caminho, janela=False): #REFATORAR DPS SE NECESSARIO
    cap = cv2.VideoCapture(caminho)

    if not cap.isOpened():
        print(f"Error: Não conseguiu abrir o video em: {caminho}")
        return [], None, 0

    print("Video aberto com sucesso!")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = round(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 0 or fps is None or fps != fps:  # fps != fps para apanhar NaN
        fps = 30

    frame_interval_ms = int(1000 / fps)
    print(f"Total frames: {frame_count}, FPS: {fps}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Fi do video alcançado ou erro na leitura do frame.")
            break
        
        frames.append(frame)
        if (janela):
            cv2.imshow("Video Frame", frame)
            tecla = cv2.waitKey(frame_interval_ms) & 0xFF
            if tecla == ord('q') or tecla == 27:
                break

    cap.release()
    cv2.destroyAllWindows()
    return frames, fps, frame_count



In [ ]:
def DefinirVistaImagem(pts, imagem):
    mascara = np.zeros(imagem.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mascara, [np.int32(pts)], 255)
    img_vista = cv2.bitwise_and(imagem, imagem, mask=mascara)
    return img_vista


def DesenharPontosVista(frame, pts_src, mostrar=False):
    frameCopia= frame.copy()

    # Desenhar os pontos
    for p in pts_src:
        cv2.circle(frameCopia, tuple(map(int, p)), 4, (0, 0, 255), -1)  # Ponto em vermelho

    # Desenhar o polígono com os pontos
    quad = np.array([pts_src[0], pts_src[1], pts_src[2], pts_src[3]], dtype=np.int32)

    cv2.polylines(frameCopia, [quad], True, color=(255, 0, 0), thickness=2)
    
    if mostrar:
        plt.imshow(cv2.cvtColor(frameCopia, cv2.COLOR_BGR2RGB))
        plt.title("imagem com pontos da vista")
        plt.grid()
        plt.show()
    return frameCopia



def CorrigirPerspetiva(frame, pts_src, altura_dst, largura_dst, mostrar=False):
    # 1. Verificar se o frame é válido
    if frame is None:
        print("Frame inválido!")
        return None
    
    altura_dst, largura_dst = frame.shape[:2]
    
    # 2. Criar os pontos destino, na mesma ordem que pts_src
    pts_dst = np.float32([
        [0, 0],                         # topo-esquerdo
        [largura_dst - 1, 0],           # topo-direito
        [0, altura_dst - 1],            # fundo-esquerdo
        [largura_dst - 1, altura_dst - 1]  # fundo-direito
    ])

    # 3. Calcular a matriz de homografia
    matriz = cv2.getPerspectiveTransform(pts_src, pts_dst)

    # 4. Aplicar a transformação de perspetiva para o tamanho desejado
    img_corrigida = cv2.warpPerspective(frame, matriz, (largura_dst, altura_dst))
    # Mostrar o resultado

    if mostrar:
        plt.imshow(cv2.cvtColor(img_corrigida, cv2.COLOR_BGR2RGB))
        plt.title("imagem corrigida da perspetiva")
        plt.grid()
        plt.show()

    return img_corrigida


In [ ]:

def EstimacaoImageFundo(caminho, n_amostras=24, guardar=False, mostrar=False):
    cap = cv2.VideoCapture(caminho)

    if not cap.isOpened():
        print("Erro: não conseguiu abrir o vídeo!")
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Selecionar frames espaçados
    indices = np.linspace(0, total - 1, n_amostras).astype(int)

    frames = []

    for i in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)

    cap.release()

    if len(frames) == 0:
        print("Erro: sem frames suficientes para calcular o fundo.")
        return None

    fundo = np.median(frames, axis=0).astype(np.uint8)

    if guardar:
        plt.imsave("fundo_estimado.jpg", fundo)
    
    if mostrar:
        plt.imshow(cv2.cvtColor(fundo, cv2.COLOR_BGR2RGB))
        plt.title("imagem do fundo estimado")
        plt.axis("off")
        plt.show()
    return fundo



def DetetarPixeisAtivos(frame, frame_fundo_estimado, limiar, mostrar=False):
    
    frameG = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    fundoG = cv2.cvtColor(frame_fundo_estimado, cv2.COLOR_BGR2GRAY)

    diff = cv2.absdiff(frameG, fundoG)

    
    clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8,8))
    diff = clahe.apply(diff)
    
    diff = cv2.medianBlur(diff, 13)
    diff = cv2.GaussianBlur(diff, (5,5), 0)

    _, bin_img = cv2.threshold(diff, limiar, 255, cv2.THRESH_BINARY)

    if mostrar:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(frameG, cmap='gray')
        plt.title("Frame Original em Cizento")
        plt.grid(True)  
        
        plt.subplot(1, 3, 2)
        plt.imshow(diff, cmap='gray')
        plt.title("Diferença entre cada frame e o fundo estimado")
        plt.grid(True)

        plt.subplot(1, 3, 3)
        plt.imshow(bin_img, cmap='gray')
        plt.title("Diferença em Binária")
        plt.grid(True)
        plt.show()

    # detetar, em cada frame, quais os píxeis que mudaram em relação ao fundo — 
    # ou seja, os píxeis que representam objetos em movimento (carros).    
    return bin_img

def Morfologicos(bin_img):
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, k_small)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9,9))
    img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, k_close)

    # Dilata ligeiro para criar contornos sólidos
    k_med = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    img = cv2.dilate(img, k_med, iterations=1)
    img = cv2.dilate(img, k_small, iterations=1)
    # Erosão para separar carros próximos
    img = cv2.erode(img, k_med, iterations=1)
    img = cv2.erode(img, k_small, iterations=1)
    
    



    return img




def DetecaoRegioesAtivas(img_bin, img_original):
    # Converter binário para formato ideal
    img = img_bin.copy()
    frame_out = img_original.copy()
    hull_out = img_original.copy()


    contornos, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hulls = [cv2.convexHull(c) for c in contornos]
    veiculos = []
    
    for c in contornos:
            area = cv2.contourArea(c)
            if area > 200:
                x, y, w, h = cv2.boundingRect(c)
                veiculos.append([x, y, w, h])
                # se quiseres desenhar aqui para debug:
                cv2.rectangle(frame_out, (x,y), (x+w, y+h), (0,255,0), 2)


    # cor_texto = (255, 255, 255)
    # tamanho_texto = 0.5
 
    # # Mostrar imagem final corretamente
    # plt.figure(figsize=(6,5))
    # plt.imshow(cv2.cvtColor(frame_out, cv2.COLOR_BGR2RGB))
    # plt.title("Veículos Detetados")
    # plt.axis("off")
    # plt.show()
    
    # ContornosHull(hull_out, hulls, contornos)

    return veiculos
    

            

def DesenharVeiculos(frame, veiculos_ativos):
    frameCopy = frame.copy()

    for v in veiculos_ativos:
        x, y, w, h, ID = v
        
        cv2.rectangle(frameCopy, (int(x), int(y)), (int(x+w), int(y+h)), (0,255,0), 2)
        cv2.putText(frameCopy, f"ID {ID}", (int(x), int(y)-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)

    return frameCopy


In [ ]:
def CorrespondenciaImagens(Boxes, veiculos, IDatual, limiar_dist=25):

    veiculo_ids = []
    centros_updated = {}
    
    for rect in Boxes:
        x, y, w, h = rect
        cx = (x + x + w)//2
        cy = (y + y + h)//2

        same_veiculo = False

        # procurar ID mais próximo
        for IDexist, (pcx, pcy) in veiculos.items():
            dist = np.hypot(cx - pcx, cy - pcy)

            if dist < limiar_dist:

                veiculos[IDexist] = (cx, cy)
                veiculo_ids.append([x, y, w, h, IDexist])
                same_veiculo = True
                break
        if not same_veiculo:
            
            # novo veículo
            veiculos[IDatual] = (cx, cy)
            veiculo_ids.append([x, y, w, h, IDatual])
            IDatual += 1

    # limpar IDs não correspondidos
    novo_dic = {}
    for v in veiculo_ids:
        _, _, _, _, idv = v
        centro = veiculos[idv]
        novo_dic[idv] = centro

    return veiculo_ids, novo_dic, IDatual


In [ ]:


def DesenharLegendaContornos(contorno, img, label, velo = 1, color=(0, 0, 0)):
    
    cv2.drawContours(img, [contorno], -1, color, 2) #Desenha os contornos na imagem original
    
    M = cv2.moments(contorno) #Obtem os momentos do contorno para calcular o centro
    cX = int(M["m10"]/M["m00"]) if M["m00"] != 0 else 0 #Calcula o centroide 
    cY = int(M["m01"]/M["m00"]) if M["m00"] != 0 else 0
    # Desenhar classificação
    cv2.putText(img, label, (cX - 20, cY), cv2.FONT_HERSHEY_SIMPLEX, 0.1 , color, 2)
    # Desenhar ratio na linha abaixo
    cv2.putText(img, f"{velo:.0f} km/h", (cX - 40, cY + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.1, color, 2)
    
#Contornos hull antes e depois
def ContornosHull(img_original, hulls, contornos):
    
    img_contornos = img_original.copy()
    img_hulls = img_original.copy()

    cv2.drawContours(img_contornos, contornos, -1, (0, 0, 0), 2)
    cv2.drawContours(img_hulls, hulls, -1, (0, 0, 255), 2)
    # Mostrar as duas imagens lado a lado
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(img_contornos, cv2.COLOR_BGR2RGB))
    plt.title("Contornos Originais")
    plt.axis("off")
    plt.subplot(1,2,2)
    plt.imshow(cv2.cvtColor(img_hulls, cv2.COLOR_BGR2RGB))
    plt.title("Contornos com Convex Hull")
    plt.axis("off")

    plt.tight_layout()
    plt.show()





def Histograma(img): #Calcula os histogramas dos três canais
    if img is None:
        print("Imagem inválida!")
        return
    
    canais = ['Blue', 'Green', 'Red']
    cores = ['b', 'g', 'r']

    plt.figure(figsize=(15, 4))

    for i, (canal, cor) in enumerate(zip(canais, cores)):
        plt.subplot(1, 3, i + 1)
        
        # Calcular histograma do canal i
        hist = cv2.calcHist([img], [i], None, [256], [0, 256])

        plt.plot(hist, color=cor)
        plt.title(f'Histograma - Canal {canal}')
        plt.xlabel('Intensidade')
        plt.ylabel('Frequência')
        plt.xlim([0, 256])
        plt.grid(True, alpha=0.5)

    plt.tight_layout()
    plt.show()



In [ ]:
#::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
#MAIN----------------------------------------------------------------------------


Ficheiro = "AutoEstrada.wm"
cap = cv2.VideoCapture(Ficheiro)

fps = round(cap.get(cv2.CAP_PROP_FPS))
intervalo_fps = int(1000 / fps)


# fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# writer = cv2.VideoWriter("resultado.mp4", fourcc, fps, (600, 800))


# topo-esquerdo 
# topo-direito 
# fundo-esquerdo 
# fundo-direito 
pts_src = np.float32([ [112, 58], 
                        [147, 58], 
                        [220, 239],
                        [12, 239] 
                        ])
#[125, 38], [140, 42], [24, 217], [220, 240] 


fundo = EstimacaoImageFundo(Ficheiro)

# fundo_corrigido = CorrigirPerspetiva(fundo, pts_src, 800, 600)
zona_detecao= DefinirVistaImagem(pts_src, fundo)

veiculos_ativos = {}  # dicionário ID -> centroide
IDatual = 0

DesenharPontosVista(fundo, pts_src, mostrar= True)


while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    frame_detecao= DefinirVistaImagem(pts_src, frame)
        
    # frame_corrigido = CorrigirPerspetiva(frame, pts_src, 800, 600)
    bin_img = DetetarPixeisAtivos(frame_detecao, zona_detecao, 100, mostrar=False)
    bin_clean = Morfologicos(bin_img)
    Boxes = DetecaoRegioesAtivas(bin_clean, frame)

    # Atualizar correspondências e obter o novo estado
    atualizado_veiculo, veiculos_ativos, IDatual = CorrespondenciaImagens(Boxes, veiculos_ativos, IDatual)

    frameVideo = DesenharVeiculos(frame, atualizado_veiculo)

    cv2.namedWindow("Contagem de Veiculos", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Contagem de Veiculos", 1080, 720)
    cv2.imshow("Contagem de Veiculos", frameVideo)
    
    cv2.imshow("Binária", bin_clean)
    cv2.imshow("Frame Detecção", frame_detecao)
    

    key = cv2.waitKey(intervalo_fps) & 0xFF
    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()



#EXTRA : METER SORT TRACKER
